<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Data Quality & Preprocessing -- 03: Data Cleaning

## Overview

Real-world data is messy. Before any analysis or modelling, you must clean it: handle missing values, fix data types, normalise text, deal with outliers, remove duplicates.

By the end of this notebook you will be able to:

- Inspect a new dataset systematically
- Identify and quantify missing values
- Choose between dropping and imputing missing data
- Convert data types correctly
- Clean and standardise text columns
- Detect and handle duplicates and outliers

**Prerequisites:** Pandas Advanced

---
# Section 1: First Inspection of a Dataset
---

Whenever you receive new data, run a checklist of inspection steps to understand its shape, types, and immediate quality issues.

<hr style="border-top:1px dashed">

## Import Libraries
We will be mainly using Pandas as well as functions from Numpy.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from config import session_datasets

<a name="anchorDataset" style="position:absolute;"></a>
<hr style="border:2px solid">

## AdventureWorks Cycles Dataset
<hr style="border-top:1px dashed">

### Do you know the AdventureWorks Cycles Dataset?
<img align="right" src="http://lh6.ggpht.com/_XjcDyZkJqHg/TPaaRcaysbI/AAAAAAAAAFo/b1U3q-qbTjY/AdventureWorks%20Logo%5B5%5D.png?imgmax=800">

Here's the Production.Product table [data dictionary](https://www.sqldatadictionary.com/AdventureWorks2014/Production.Product.html), which is a description of the fields (columns) in the table (the .csv file we will import below):<br>
- **ProductID** - Primary key for Product records.
- **Name** - Name of the product.
- **ProductNumber** - Unique product identification number.
- **MakeFlag** - 0 = Product is purchased, 1 = Product is manufactured in-house.
- **FinishedGoodsFlag** - 0 = Product is not a salable item. 1 = Product is salable.
- **Color** - Product color.
- **SafetyStockLevel** - Minimum inventory quantity.
- **ReorderPoint** - Inventory level that triggers a purchase order or work order.
- **StandardCost** - Standard cost of the product.
- **ListPrice** - Selling price.
- **Size** - Product size.
- **SizeUnitMeasureCode** - Unit of measure for the Size column.
- **WeightUnitMeasureCode** - Unit of measure for the Weight column.
- **DaysToManufacture** - Number of days required to manufacture the product.
- **ProductLine** - R = Road, M = Mountain, T = Touring, S = Standard
- **Class** - H = High, M = Medium, L = Low
- **Style** - W = Womens, M = Mens, U = Universal
- **ProductSubcategoryID** - Product is a member of this product subcategory. Foreign key to ProductSubCategory.ProductSubCategoryID.
- **ProductModelID** - Product is a member of this product model. Foreign key to ProductModel.ProductModelID.
- **SellStartDate** - Date the product was available for sale.
- **SellEndDate** - Date the product was no longer available for sale.
- **DiscontinuedDate** - Date the product was discontinued.
- **rowguid** - ROWGUIDCOL number uniquely identifying the record. Used to support a merge replication sample.
- **ModifiedDate** - Date and time the record was last updated.


<a name="anchorInitialImpressions" style="position:absolute;"></a>
<hr style="border:2px solid">

## Initial Impressions
<hr style="border-top:1px dashed">

For our data, we are going to use the `read_csv()` function. 

<div class="alert alert-block alert-warning">
    <b>Question:</b> What does CSV stand for?
</div>

If you look at our data file, you can see it is separated by tabs instead. When you type out `.read_csv()` press `Shift+Tab` while having your cursor in the parenthesis. One of the parameters we can use to help us is called `sep=`

That being said, we are going to use the `'\t'` separator to specify tab-delimited columns.

In [2]:
##Local files
#prod = pd.read_csv('../datasets/Production.Product.csv', sep='\t')

#S3:
prod = pd.read_csv(session_datasets["Production.Product"], sep='\t')


In [ ]:
"""
#Databricks:
# S3:
prod = (
    spark.read.csv(
        session_datasets["Production.Product"],
        header=True,
        inferSchema=True,
        sep="\t"
    )
).toPandas()
"""

Whenever we have a new dataset, it's always a good idea to look at the top couple of rows once loaded in.

Let's check out the first 3 rows using `.head()`

In [3]:
prod.head(3)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
0,1,Adjustable Race,AR-5381,0,0,NaN,1000,750,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{694215B7-08F7-4C0D-ACB1-D734BA44C0C8},2014-02-08 10:01:36.827000000
1,2,Bearing Ball,BA-8327,0,0,NaN,1000,750,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{58AE3C20-4F3A-4749-A7D4-D568806CC537},2014-02-08 10:01:36.827000000
2,3,BB Ball Bearing,BE-2349,1,0,NaN,800,600,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{9C21AED2-5BFA-4F18-BCB8-F11638DC2E4E},2014-02-08 10:01:36.827000000


If we instead wanted to take a look at the last couple of rows, we would use `tail()`.

In [5]:
prod.tail(4)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
500,996,HL Bottom Bracket,BB-9108,1,1,NaN,500,375,53.9416,121.49,...,NaN,H,NaN,5.0,97.0,2013-05-30 00:00:00,NaN,NaN,{230C47C5-08B2-4CE3-B706-69C0BDD62965},2014-02-08 10:01:36.827000000
501,997,"Road-750 Black, 44",BK-R19B-44,1,1,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30 00:00:00,NaN,NaN,{44CE4802-409F-43AB-9B27-CA53421805BE},2014-02-08 10:01:36.827000000
502,998,"Road-750 Black, 48",BK-R19B-48,1,1,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30 00:00:00,NaN,NaN,{3DE9A212-1D49-40B6-B10A-F564D981DBDE},2014-02-08 10:01:36.827000000
503,999,"Road-750 Black, 52",BK-R19B-52,1,1,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30 00:00:00,NaN,NaN,{AE638923-2B67-4679-B90E-ABBAB17DCA31},2014-02-08 10:01:36.827000000


In [6]:
prod.sample(5)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
157,479,Metal Plate 2,MP-2066,0,0,NaN,1000,750,0.0000,0.00,...,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{0773A2C9-F47F-429E-814A-25B2E08C128A},2014-02-08 10:01:36.827000000
446,942,"ML Mountain Frame-W - Silver, 38",FR-M63S-38,1,1,Silver,500,375,199.3757,364.09,...,M,M,W,12.0,15.0,2013-05-30 00:00:00,NaN,NaN,{BA3646B0-1487-426E-AB4E-57D42E6F9233},2014-02-08 10:01:36.827000000
143,465,Lock Washer 10,LW-1201,0,0,NaN,1000,750,0.0000,0.00,...,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{A2212BAB-AF58-41A5-A659-A6141C8967CA},2014-02-08 10:01:36.827000000
278,774,"Mountain-100 Silver, 48",BK-M82S-48,1,1,Silver,100,75,1912.1544,3399.99,...,M,H,U,1.0,19.0,2011-05-31 00:00:00,2012-05-29 00:00:00,NaN,{BA5551DF-C9EE-4B43-B3CA-8C19D0F9384D},2014-02-08 10:01:36.827000000
484,980,"Mountain-400-W Silver, 38",BK-M38S-38,1,1,Silver,100,75,419.7784,769.49,...,M,M,W,1.0,22.0,2013-05-30 00:00:00,NaN,NaN,{7A927632-99A4-4F24-ADCE-0062D2D113D9},2014-02-08 10:01:36.827000000


Next, we have several other functions that let us take a look at the dataframe's aspects
- `.shape`
- `.dtypes`
- `.info()`
- `.describe()`
- `.columns`
- `.unique()`
- `.nunique()`
- `.unique()`
- `.value_counts()`

Let's explore them real quick:

In [7]:
prod.shape # 504 rows and 25 cols, but not all cols show up in head and tail function. 

(504, 25)

In [8]:
pd.set_option('display.max_columns', 25) # https://pandas.pydata.org/docs/reference/api/pandas.set_option.html
prod.head(3)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
0,1,Adjustable Race,AR-5381,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{694215B7-08F7-4C0D-ACB1-D734BA44C0C8},2014-02-08 10:01:36.827000000
1,2,Bearing Ball,BA-8327,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{58AE3C20-4F3A-4749-A7D4-D568806CC537},2014-02-08 10:01:36.827000000
2,3,BB Ball Bearing,BE-2349,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{9C21AED2-5BFA-4F18-BCB8-F11638DC2E4E},2014-02-08 10:01:36.827000000


In [9]:
print(prod.columns)

Index(['ProductID', 'Name', 'ProductNumber', 'MakeFlag', 'FinishedGoodsFlag',
       'Color', 'SafetyStockLevel', 'ReorderPoint', 'StandardCost',
       'ListPrice', 'Size', 'SizeUnitMeasureCode', 'WeightUnitMeasureCode',
       'Weight', 'DaysToManufacture', 'ProductLine', 'Class', 'Style',
       'ProductSubcategoryID', 'ProductModelID', 'SellStartDate',
       'SellEndDate', 'DiscontinuedDate', 'rowguid', 'ModifiedDate'],
      dtype='object')


<div class="alert alert-block alert-info">
    <b>Columns can be misread as "objects" when they are really floats. If that is the case, use <code>astype()</code> to cast a column as a different type.</b>
</div>

In [10]:
prod.dtypes

ProductID                  int64
Name                      object
ProductNumber             object
MakeFlag                   int64
FinishedGoodsFlag          int64
Color                     object
SafetyStockLevel           int64
ReorderPoint               int64
StandardCost             float64
ListPrice                float64
Size                      object
SizeUnitMeasureCode       object
WeightUnitMeasureCode     object
Weight                   float64
DaysToManufacture          int64
ProductLine               object
Class                     object
Style                     object
ProductSubcategoryID     float64
ProductModelID           float64
SellStartDate             object
SellEndDate               object
DiscontinuedDate         float64
rowguid                   object
ModifiedDate              object
dtype: object

In [11]:
prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ProductID              504 non-null    int64  
 1   Name                   504 non-null    object 
 2   ProductNumber          504 non-null    object 
 3   MakeFlag               504 non-null    int64  
 4   FinishedGoodsFlag      504 non-null    int64  
 5   Color                  256 non-null    object 
 6   SafetyStockLevel       504 non-null    int64  
 7   ReorderPoint           504 non-null    int64  
 8   StandardCost           504 non-null    float64
 9   ListPrice              504 non-null    float64
 10  Size                   211 non-null    object 
 11  SizeUnitMeasureCode    176 non-null    object 
 12  WeightUnitMeasureCode  205 non-null    object 
 13  Weight                 205 non-null    float64
 14  DaysToManufacture      504 non-null    int64  
 15  Produc

In [12]:
prod.describe() # only for numerical values - be careful with some numerical values that are actually categorical such as MakeFlag

,ProductID,MakeFlag,FinishedGoodsFlag,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Weight,DaysToManufacture,ProductSubcategoryID,ProductModelID,DiscontinuedDate
count,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,205.000000,504.000000,295.000000,295.000000,0.0
mean,673.039683,0.474206,0.585317,535.150794,401.363095,258.602961,438.666250,74.069220,1.103175,12.294915,37.444068,NaN
std,229.373142,0.499830,0.493157,374.112954,280.584715,461.632808,773.602843,182.166588,1.492616,9.860135,34.025442,NaN
min,1.000000,0.000000,0.000000,4.000000,3.000000,0.000000,0.000000,2.120000,0.000000,1.000000,1.000000,NaN
25%,447.750000,0.000000,0.000000,100.000000,75.000000,0.000000,0.000000,2.880000,0.000000,2.000000,11.000000,NaN
50%,747.500000,0.000000,1.000000,500.000000,375.000000,23.372200,49.990000,17.900000,1.000000,12.000000,26.000000,NaN
75%,873.250000,1.000000,1.000000,1000.000000,750.000000,317.075825,564.990000,27.350000,1.000000,17.000000,48.500000,NaN
max,999.000000,1.000000,1.000000,1000.000000,750.000000,2171.294200,3578.270000,1050.000000,4.000000,37.000000,128.000000,NaN


<div class="alert alert-block alert-success">
    <b>Using these functions together at the start of EDA can save hours if not days of time.
</div>

Did you notice anything special about `ProductID`? Use the function `nunique()` to count the distinct values over our `ProductID` column.

In [13]:
prod['ProductID'].nunique()

504

Since our shape says there are 504 columns, and there are 504 unique values in this column (as well as the data dictionary telling us that it is the primary key) it is safe to say that this is indeed the primary key.

That being the case, let's bring our `ProductID` column into the index since it's the PK (primary key) of our table and that's where PKs belong as a best practice.

<div class="alert alert-block alert-warning">
    <b>Question:</b> How would we move the <code>ProductID</code> column into the index column?
</div>

In [14]:
#prod.set_index('ProductID', inplace=True)

In [15]:
prod = prod.set_index('ProductID')

Using `.head()`, let's make sure that worked and there aren't any extra parameters \*cough cough\* that we might need to change:

In [16]:
prod.head(10)

,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
ProductID,,,,,,,,,,,,,,,,,,,,,,,,
1,Adjustable Race,AR-5381,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{694215B7-08F7-4C0D-ACB1-D734BA44C0C8},2014-02-08 10:01:36.827000000
2,Bearing Ball,BA-8327,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{58AE3C20-4F3A-4749-A7D4-D568806CC537},2014-02-08 10:01:36.827000000
3,BB Ball Bearing,BE-2349,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{9C21AED2-5BFA-4F18-BCB8-F11638DC2E4E},2014-02-08 10:01:36.827000000
4,Headset Ball Bearings,BE-2908,0,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{ECFED6CB-51FF-49B5-B06C-7D8AC834DB8B},2014-02-08 10:01:36.827000000
316,Blade,BL-2036,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{E73E9750-603B-4131-89F5-3DD15ED5FF80},2014-02-08 10:01:36.827000000
317,LL Crankarm,CA-5965,0,0,Black,500,375,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,L,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{3C9D10B7-A6B2-4774-9963-C19DCEE72FEA},2014-02-08 10:01:36.827000000
318,ML Crankarm,CA-6738,0,0,Black,500,375,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,M,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{EABB9A92-FA07-4EAB-8955-F0517B4A4CA7},2014-02-08 10:01:36.827000000
319,HL Crankarm,CA-7457,0,0,Black,500,375,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{7D3FD384-4F29-484B-86FA-4206E276FE58},2014-02-08 10:01:36.827000000
320,Chainring Bolts,CB-2903,0,0,Silver,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{7BE38E48-B7D6-4486-888E-F53C26735101},2014-02-08 10:01:36.827000000


---
# Section 2: Identifying Missing Data
---

Missing values appear as `NaN` (Not a Number) in Pandas. Two key methods:

- **`.isnull()` / `.isna()`** -- returns a boolean DataFrame
- **`.notnull()` / `.notna()`** -- the opposite

Combine with `.sum()` to count missing per column.

## Handling Missing Data


Recall missing data is a systemic, challenging problem for data scientists. Imagine conducting a poll, but some of the data gets lost, or you run out of budget and can't complete it! 

"Handling missing data" itself is a broad topic. We'll focus on two components:

- Using Pandas to identify we have missing data
- Strategies to fill in missing data (known in the business as `imputing`)
- Filling in missing data with Pandas


___
### Identifying missing data

Before *handling*, we must identify we're missing data at all!

We have a few ways to explore missing data, and they are reminiscient of our Boolean filters.

In [17]:
# True when data isn't missing
prod.notnull().head(3)

,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
ProductID,,,,,,,,,,,,,,,,,,,,,,,,
1,True,True,True,True,False,True,True,True,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,True
2,True,True,True,True,False,True,True,True,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,True
3,True,True,True,True,False,True,True,True,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,True


In [18]:
# True when data is missing
prod.isnull().head(3)

,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
ProductID,,,,,,,,,,,,,,,,,,,,,,,,
1,False,False,False,False,True,False,False,False,False,True,True,True,True,False,True,True,True,True,True,False,True,True,False,False
2,False,False,False,False,True,False,False,False,False,True,True,True,True,False,True,True,True,True,True,False,True,True,False,False
3,False,False,False,False,True,False,False,False,False,True,True,True,True,False,True,True,True,True,True,False,True,True,False,False


Now, we may want to see null values in aggregate. We can use `sum()` to sum down a given column

In [19]:
# here is a quick way to do it
prod.isnull().sum()

Name                       0
ProductNumber              0
MakeFlag                   0
FinishedGoodsFlag          0
Color                    248
SafetyStockLevel           0
ReorderPoint               0
StandardCost               0
ListPrice                  0
Size                     293
SizeUnitMeasureCode      328
WeightUnitMeasureCode    299
Weight                   299
DaysToManufacture          0
ProductLine              226
Class                    257
Style                    293
ProductSubcategoryID     209
ProductModelID           209
SellStartDate              0
SellEndDate              406
DiscontinuedDate         504
rowguid                    0
ModifiedDate               0
dtype: int64


<div class="alert alert-block alert-danger">
    <b>Look! We've found missing values!</b>
</div>

<div class="alert alert-block alert-warning">
    <b>Question:</b> How could this missing data be problematic for our analysis?
</div>


<div class="alert alert-block alert-danger">
    <b>DataSet Null Characters:</b> Did you find no nulls in your data? <br>Some datasets will come with other characters instead of <code>Nulls</code> you can use the <code>.replace()</code> to replace those characters with <code>np.nan</code> instead.<br> It is almost impossible to find perfect Data, better to check again then miss null characters like a <code>?</code> acting as an invisible null.
</div>

___
### Understanding missing data

Finding missing data is the easy part! Determining way to do next is more complicated.

Typically, we are most interested in knowing **why** we have missing data. Once we know what 'type of missingness' we have (the source of missing data), we can proceed effectively.

Let's first quantify how much data we are missing. Here is another implementation of `prod.isnull().sum()`, only wrapped with a `DataFrame` and some labels to make it a little more user-friendly:

In [20]:
# or we can make things pretty as follows
null_df = pd.DataFrame(prod.isnull().sum(), columns=['Count of Nulls']) #Creating a new DF

null_df.index.name = 'Column' # Changing the Index Name

null_df.sort_values(['Count of Nulls'], ascending=False) # Sorting based on our only column name

,Count of Nulls
Column,
DiscontinuedDate,504
SellEndDate,406
SizeUnitMeasureCode,328
Weight,299
WeightUnitMeasureCode,299
Size,293
Style,293
Class,257
Color,248


---
# Section 3: Handling Missing Values
---

You have two main choices for missing data:

| Strategy | When to use |
|----------|-------------|
| **Drop** (`.dropna()`) | The missing rows/columns are few, or the data is critical |
| **Fill / Impute** (`.fillna()`) | You have many missing values and dropping would waste data |

**Common imputation strategies:**
- Numerical: mean, median, or 0
- Categorical: mode (most common value), or a sentinel like `'Unknown'`
- Time series: forward fill (`ffill`), backward fill (`bfill`), interpolation

___
### Filling in missing data

How we fill in data depends largely on why it is missing (types of missingness) and what sampling we have available to us.

We may:

- Delete missing data altogether
- Fill in missing data with: (also called data imputation)
    - The average of the column
    - The median of the column (the middle number)
    - A predicted amount based on other factors
- Collect more data:
    - Resample the population
    - Followup with the authority providing data that is missing
    
  
 Reference Material: https://www.datacamp.com/tutorial/techniques-to-handle-missing-data-values


In our case, let's focus on handling missing values in `Color`. Let's get a count of the unique values in that column. We will need to use the `dropna=False` kwarg, otherwise the `pd.Series.value_counts()` method will not count `NaN` (null) values.

In [21]:
prod['Color'].value_counts(dropna=False)

Color
NaN             248
Black            93
Silver           43
Red              38
Yellow           36
Blue             26
Multi             8
Silver/Black      7
White             4
Grey              1
Name: count, dtype: int64

### **Option 1: Drop the missing values.**

In [22]:
# drops rows where any row has a missing value - this does not happen *in place*, so we are not actually dropping
type(prod['Color'].dropna(inplace=False))

pandas.core.series.Series

In [23]:
#if I wanted to see the impact of above
prod['Color'].dropna(inplace=False)

ProductID
317     Black
318     Black
319     Black
320    Silver
321    Silver
        ...  
992     Black
993     Black
997     Black
998     Black
999     Black
Name: Color, Length: 256, dtype: object

**Important!** `pd.DataFrame.dropna()` and `pd.Series.dropna()` are very versatile! Let's look at the docs (Series is similar):

```python
Signature: pd.DataFrame.dropna(self, axis=0, how='any', thresh=None, subset=None, inplace=False)
Docstring:
Remove missing values.

See the :ref:`User Guide <missing_data>` for more on which values are
considered missing, and how to work with missing data.

Parameters
----------
axis : {0 or 'index', 1 or 'columns'}, default 0
    Determine if rows or columns which contain missing values are
    removed.

    * 0, or 'index' : Drop rows which contain missing values.
    * 1, or 'columns' : Drop columns which contain missing value.

    .. deprecated:: 0.23.0: Pass tuple or list to drop on multiple
    axes.
how : {'any', 'all'}, default 'any'
    Determine if row or column is removed from DataFrame, when we have
    at least one NA or all NA.

    * 'any' : If any NA values are present, drop that row or column.
    * 'all' : If all values are NA, drop that row or column.
thresh : int, optional
    Require that many non-NA values.
subset : array-like, optional
    Labels along other axis to consider, e.g. if you are dropping rows
    these would be a list of columns to include.
inplace : bool, default False
    If True, do operation inplace and return None.
```

**how**: This tells us if we want to remove a row if _any_ of the columns have a null, or _all_ of the columns have a null.<br>
**subset**: We can input an array here, like `['Color', 'Size', 'Weight']`, and it will only consider nulls in those columns. This is very useful!<br>
**inplace**: This is if you want to mutate (change) the source dataframe. Default is `False`, so it will return a _copy_ of the source dataframe.

**To accomplish the same thing, but implement it on our entire dataframe, we can do the following:**

In [24]:
# drops all nulls from the Color column, but returns the entire dataframe instead of just the Color column
prod.dropna(subset=['Color']).head()

Column,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
ProductID,,,,,,,,,,,,,,,,,,,,,,,,
317,LL Crankarm,CA-5965,0,0,Black,500,375,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,L,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{3C9D10B7-A6B2-4774-9963-C19DCEE72FEA},2014-02-08 10:01:36.827000000
318,ML Crankarm,CA-6738,0,0,Black,500,375,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,M,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{EABB9A92-FA07-4EAB-8955-F0517B4A4CA7},2014-02-08 10:01:36.827000000
319,HL Crankarm,CA-7457,0,0,Black,500,375,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{7D3FD384-4F29-484B-86FA-4206E276FE58},2014-02-08 10:01:36.827000000
320,Chainring Bolts,CB-2903,0,0,Silver,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{7BE38E48-B7D6-4486-888E-F53C26735101},2014-02-08 10:01:36.827000000
321,Chainring Nut,CN-6137,0,0,Silver,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2008-04-30 00:00:00,NaN,NaN,{3314B1D7-EF69-4431-B6DD-DC75268BD5DF},2014-02-08 10:01:36.827000000


### **Option 2: Fill in missing values**

Traditionally, we fill missing data with a median (midpoint value), average, or mode (most frequently occurring). For `Color`, let's replace the nulls with the string value `NoColor`.

Let's first look at the way we'd do it with a single column, using the `pd.Series.fillna()` method:

In [25]:
prod['Color'].value_counts()

Color
Black           93
Silver          43
Red             38
Yellow          36
Blue            26
Multi            8
Silver/Black     7
White            4
Grey             1
Name: count, dtype: int64

In [26]:
prod['Color'].fillna('NoColor')

ProductID
1      NoColor
2      NoColor
3      NoColor
4      NoColor
316    NoColor
        ...   
995    NoColor
996    NoColor
997      Black
998      Black
999      Black
Name: Color, Length: 504, dtype: object

Now let's store that into a new column called *nColor* which stands for **new**. From now on, every time we do any pre-processing we will apply that to a new column:

In [27]:
prod['nColor'] = prod['Color'].fillna('NoColor') # store that in a new col called nColor

In [28]:
prod[['Color', 'nColor']].head()

Column,Color,nColor
ProductID,,
1,NaN,NoColor
2,NaN,NoColor
3,NaN,NoColor
4,NaN,NoColor
316,NaN,NoColor


In [29]:
prod['nColor'].value_counts()

nColor
NoColor         248
Black            93
Silver           43
Red              38
Yellow           36
Blue             26
Multi             8
Silver/Black      7
White             4
Grey              1
Name: count, dtype: int64

In [30]:
## We can also fillup the missing values with mode
print(prod['Color'].value_counts(dropna=False))
color_mode = prod['Color'].mode().item()
prod["New_Color_Mode"] = prod['Color'].fillna(color_mode)
print(prod['New_Color_Mode'].value_counts(dropna=False))

Color
NaN             248
Black            93
Silver           43
Red              38
Yellow           36
Blue             26
Multi             8
Silver/Black      7
White             4
Grey              1
Name: count, dtype: int64
New_Color_Mode
Black           341
Silver           43
Red              38
Yellow           36
Blue             26
Multi             8
Silver/Black      7
White             4
Grey              1
Name: count, dtype: int64


**Exercise**: Do the same for the Class, Style, ProductLine columns by filling NAs with noClass, noStyle,noProductLine. 

In [31]:
prod['nClass'] = prod['Class'].fillna('NoClass')
prod['nClass'].value_counts()


nClass
NoClass    257
L           97
H           82
M           68
Name: count, dtype: int64

The other col we might want to fill NAs is Style and Product Line:

In [32]:
prod.fillna(value = {'Style': 'NoStyle',
                     'ProductLine': 'NoLine'})[['Style', 'ProductLine']]

Column,Style,ProductLine
ProductID,,
1,NoStyle,NoLine
2,NoStyle,NoLine
3,NoStyle,NoLine
4,NoStyle,NoLine
316,NoStyle,NoLine
...,...,...
995,NoStyle,NoLine
996,NoStyle,NoLine
997,U,R


In [33]:
prod['nStyle'] = prod['Style'].fillna('NoStyle')
prod['nStyle'].value_counts()

nStyle
NoStyle    293
U          176
W           28
M            7
Name: count, dtype: int64

In [34]:
prod['nProductLine'] = prod['ProductLine'].fillna('NoLine')
prod['nProductLine'].value_counts()

nProductLine
NoLine    226
R         100
M          91
T          52
S          35
Name: count, dtype: int64

Additionally, we can reference any other data or formulas we want with the value we fill the nulls with. This is very handy if you want to impute with the average or median of that column... or even another column altogether! Here is an example where we will replace the nulls of `Weight` with the average value from the `Weight` column.

In [35]:
##Numerical
##Working with "weight"
##Mean:
print(prod['Weight'].value_counts(dropna=False))
weight_mean = prod['Weight'].mean().item()
prod["New_Weight_Mean"] = prod['Weight'].fillna(weight_mean)
print(prod['New_Weight_Mean'].value_counts(dropna=False))

##Default value:
prod["New_Weight"] = prod['Weight'].fillna(1)
print(prod['New_Weight'].value_counts(dropna=False))

Weight
NaN      299
2.30       4
2.96       4
3.00       4
3.04       4
        ... 
18.42      1
18.13      1
17.77      1
17.35      1
20.42      1
Name: count, Length: 128, dtype: int64
New_Weight_Mean
74.06922    299
2.30000       4
2.96000       4
3.00000       4
3.04000       4
           ... 
18.42000      1
18.13000      1
17.77000      1
17.35000      1
20.42000      1
Name: count, Length: 128, dtype: int64
New_Weight
1.00     299
2.30       4
2.96       4
3.00       4
3.04       4
        ... 
18.42      1
18.13      1
17.77      1
17.35      1
20.42      1
Name: count, Length: 128, dtype: int64


In [36]:
prod['nWeight'] = prod['Weight'].fillna(prod['Weight'].mean())
prod[['Weight', 'nWeight']]

Column,Weight,nWeight
ProductID,,
1,NaN,74.06922
2,NaN,74.06922
3,NaN,74.06922
4,NaN,74.06922
316,NaN,74.06922
...,...,...
995,168.00,168.00000
996,170.00,170.00000
997,19.77,19.77000


**Exercise:** Now, use the reviewed  techniques to fill NAs from:
- Size
- SizeUnitMeasureCode

In [37]:
prod['Size'].value_counts()
#prod['Size'].astype(float) # can't do that because sizes can be integers and strings!!!
#prod['Size'].fillna(prod['Size'].mean())
prod['nSize'] = prod['Size'].fillna('NoSize')
prod[['nSize', 'Size']]

Column,nSize,Size
ProductID,,
1,NoSize,NaN
2,NoSize,NaN
3,NoSize,NaN
4,NoSize,NaN
316,NoSize,NaN
...,...,...
995,NoSize,NaN
996,NoSize,NaN
997,44,44


In [38]:
prod['Size'].value_counts()


Size
44    29
48    25
52    16
58    15
42    15
38    12
L     11
40    11
62    11
M     11
60    11
46    11
50     9
54     9
S      9
XL     3
56     2
70     1
Name: count, dtype: int64

In [39]:
prod['SizeUnitMeasureCode'].value_counts()

SizeUnitMeasureCode
CM     176
Name: count, dtype: int64

In [40]:
prod['nSizeUnitMeasureCode'] = prod['SizeUnitMeasureCode'].fillna('noUnit')
prod[['SizeUnitMeasureCode', 'nSizeUnitMeasureCode']]

Column,SizeUnitMeasureCode,nSizeUnitMeasureCode
ProductID,,
1,NaN,noUnit
2,NaN,noUnit
3,NaN,noUnit
4,NaN,noUnit
316,NaN,noUnit
...,...,...
995,NaN,noUnit
996,NaN,noUnit
997,CM,CM


---
# Section 4: Data Type Conversion
---

Pandas often infers types correctly, but sometimes needs help. The most common conversions:

```python
df['col'] = df['col'].astype(int)       # to integer
df['col'] = df['col'].astype(float)     # to float
df['col'] = df['col'].astype(str)       # to string
df['col'] = pd.to_datetime(df['col'])   # to datetime
df['col'] = pd.to_numeric(df['col'], errors='coerce')  # invalid -> NaN
```

In [41]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'price_str': ['10.50', '25.99', 'invalid', '7.00', None],
    'date_str':  ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05', '2024-05-01'],
    'count':     ['100', '200', '50', '75', '300']
})
print(df.dtypes)

# Convert price (with bad values -> NaN)
df['price'] = pd.to_numeric(df['price_str'], errors='coerce')

# Convert dates
df['date'] = pd.to_datetime(df['date_str'])

# Convert count to int
df['count'] = df['count'].astype(int)

print(df.dtypes)
df

price_str    object
date_str     object
count        object
dtype: object
price_str            object
date_str             object
count                 int64
price               float64
date         datetime64[ns]
dtype: object


,price_str,date_str,count,price,date
0,10.50,2024-01-15,100,10.50,2024-01-15
1,25.99,2024-02-20,200,25.99,2024-02-20
2,invalid,2024-03-10,50,NaN,2024-03-10
3,7.00,2024-04-05,75,7.00,2024-04-05
4,None,2024-05-01,300,NaN,2024-05-01


---
# Section 5: Transforming Values with `.apply()`
---

`.apply()` is the bridge between Pandas and arbitrary Python -- use it to apply custom logic to a column. Especially useful for cleaning rules (e.g. mapping codes to names).

<a name="anchorApply" style="position:absolute;"></a>
<hr style="border:2px solid">

## Apply Function
<hr style="border-top:1px dashed">

Apply functions allow us to perform a complex operation across an entire column efficiently.

For example, let's say we want to change our colors from a word, to just a single letter. How would we do that?

The first step is writing a function, with the argument being the value we would receive from each cell in the column. This function will mutate the input, and return the result. This result will then be _applied_ to the source dataframe (if desired).

In [42]:
prod['nColor'].unique()

array(['NoColor', 'Black', 'Silver', 'Red', 'White', 'Blue', 'Multi',
       'Yellow', 'Grey', 'Silver/Black'], dtype=object)

In [43]:
def color_to_letter(color):
    color_dict = {
        'Black': 'B', 
        'Silver': 'S', 
        'Red': 'R', 
        'White': 'W', 
        'Blue': 'Bl', 
        'Multi': 'M', 
        'Yellow': 'Y',
        'Grey': 'G', 
        'Silver/Black': 'SB',
        'NoColor': 'NC'
    }
    
#     list_of_keys = color_dict.keys()
    
#     if color in list_of_keys:
#         return color_dict[color]
#     else:
#         return 'N'
    
    try:
        return color_dict[color] # Try to run >> has error >> Except
    except:
        return 'N' #An Error

Now we can _apply_ this function to our `pd.Series` object, returning the result (which we can use to overwrite the source, if we choose).

In [44]:
prod['nColor'].apply(color_to_letter).head(10)
prod['nColor']

ProductID
1      NoColor
2      NoColor
3      NoColor
4      NoColor
316    NoColor
        ...   
995    NoColor
996    NoColor
997      Black
998      Black
999      Black
Name: nColor, Length: 504, dtype: object

The `pd.DataFrame.apply` implementation is similar, however it effectively 'scrolls through' the columns and passes each one sequentially to your function:

```python
Objects passed to the function are Series objects whose index is
either the DataFrame's index (``axis=0``) or the DataFrame's columns
(``axis=1``).
```

It should only be used when you wish to apply the same function to all columns (or rows) of your `pd.DataFrame` object.

We can also use `pd.Series.apply()` with a **labmda expression**. This is an undeclared function and is commonly used for simple functions within the `.apply()` method. Let's use it to add $100 to our `ListPrice` column. 

In [45]:
# without apply
prod['ListPrice'].tail(10)

ProductID
990    539.99
991    539.99
992    539.99
993    539.99
994     53.99
995    101.24
996    121.49
997    539.99
998    539.99
999    539.99
Name: ListPrice, dtype: float64

In [46]:
# and now with 100 more dollars!
prod['ListPrice'].apply(lambda x: x+100).tail(10)

ProductID
990    639.99
991    639.99
992    639.99
993    639.99
994    153.99
995    201.24
996    221.49
997    639.99
998    639.99
999    639.99
Name: ListPrice, dtype: float64

**Exercise** Use the lambda function to double the SafetyStockLevel

In [47]:
prod['nSafetyStockLevel'] = prod['SafetyStockLevel'].apply(lambda x: x*2).head()
prod.head()

Column,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,...,ModifiedDate,nColor,New_Color_Mode,nClass,nStyle,nProductLine,New_Weight_Mean,New_Weight,nWeight,nSize,nSizeUnitMeasureCode,nSafetyStockLevel
ProductID,,,,,,,,,,,,,,,,,,,,,,,,,
1,Adjustable Race,AR-5381,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,...,2014-02-08 10:01:36.827000000,NoColor,Black,NoClass,NoStyle,NoLine,74.06922,1.0,74.06922,NoSize,noUnit,2000.0
2,Bearing Ball,BA-8327,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,...,2014-02-08 10:01:36.827000000,NoColor,Black,NoClass,NoStyle,NoLine,74.06922,1.0,74.06922,NoSize,noUnit,2000.0
3,BB Ball Bearing,BE-2349,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,...,2014-02-08 10:01:36.827000000,NoColor,Black,NoClass,NoStyle,NoLine,74.06922,1.0,74.06922,NoSize,noUnit,1600.0
4,Headset Ball Bearings,BE-2908,0,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,...,2014-02-08 10:01:36.827000000,NoColor,Black,NoClass,NoStyle,NoLine,74.06922,1.0,74.06922,NoSize,noUnit,1600.0
316,Blade,BL-2036,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,...,2014-02-08 10:01:36.827000000,NoColor,Black,NoClass,NoStyle,NoLine,74.06922,1.0,74.06922,NoSize,noUnit,1600.0


Now let's make a function that will replace the current values using the following dictionaries (and save it to the new column)
- ProductLine - R = Road, M = Mountain, T = Touring, S = Standard
- Class - H = High, M = Medium, L = Low
- Style - W = Womens, M = Mens, U = Universal

In [48]:
def product_line(line):
    line_dict = {
        'R': 'Road', 
        'M': 'Mountain', 
        'T': 'Touring', 
        'S': 'Standard',
        'NoLine': 'NoLine'
    }
    
    list_of_keys = line_dict.keys()
    
    if line in list_of_keys:
        return line_dict[line]
    else:
         return 'N'
        
prod['nProductLine'].apply(product_line)
#prod['nProductLine']

ProductID
1      NoLine
2      NoLine
3      NoLine
4      NoLine
316    NoLine
        ...  
995    NoLine
996    NoLine
997         N
998         N
999         N
Name: nProductLine, Length: 504, dtype: object

In [49]:
prod['nProductLine'].value_counts()

nProductLine
NoLine    226
R         100
M          91
T          52
S          35
Name: count, dtype: int64

In [50]:
prod['nProductLine'].unique() # it is not working because our values have added spaces!!!

array(['NoLine', 'R ', 'S ', 'M ', 'T '], dtype=object)

---
# Section 6: Cleaning Text Columns
---

Text data is one of the dirtiest sources -- whitespace, mixed case, typos, accents. Pandas exposes string methods via `.str`:

```python
df['name'] = df['name'].str.strip()       # remove whitespace
df['name'] = df['name'].str.lower()       # lowercase
df['name'] = df['name'].str.replace('-', ' ')  # substitution
df['name'] = df['name'].str.title()       # title case
```

<a name="anchorStringPreprocessing" style="position:absolute;"></a>
<hr style="border:2px solid">

## String pre-processing
<hr style="border-top:1px dashed">


In [51]:
prod['nProductLine'] = prod['nProductLine'].str.strip()
prod['nProductLine'].unique()

array(['NoLine', 'R', 'S', 'M', 'T'], dtype=object)

In [52]:
#check if function works now
prod['nProductLine'] = prod['nProductLine'].apply(product_line)
prod['nProductLine'].unique()

array(['NoLine', 'Road', 'Standard', 'Mountain', 'Touring'], dtype=object)

Now your job is to do the same to the nClass column:
- remove additional spaces
- apply a function that will replace High for H, Medium for M, and Low for L

In [53]:
prod['nClass']= prod['nClass'].str.strip()
prod['nClass'].unique()

array(['NoClass', 'L', 'M', 'H'], dtype=object)

In [54]:
def product_class(clas):
    class_dict = {
        'L': 'Low', 
        'M': 'Medium', 
        'H': 'High', 
        'NoClass': 'NoClass'
    }
    
    list_of_keys = class_dict.keys()
    
    if clas in list_of_keys:
        return class_dict[clas]
    else:
         return 'N'
        
prod['nClass'].apply(product_class)
#prod['nProductLine']

ProductID
1      NoClass
2      NoClass
3      NoClass
4      NoClass
316    NoClass
        ...   
995     Medium
996       High
997        Low
998        Low
999        Low
Name: nClass, Length: 504, dtype: object

In [55]:
prod['nClass'] = prod['Class'].str.strip()
prod['nClass'] = prod['nClass'].fillna('noClass')
prod['nClass'].unique()

array(['noClass', 'L', 'M', 'H'], dtype=object)

The same can also be achieved with the **map** and **replace** functions:

In [56]:
class_dict = {
        'L': 'Low', 
        'M': 'Medium', 
        'H': 'High', 
        'noClass': 'noClass'
    }
prod['mapClass'] = prod['nClass'].map(class_dict)
prod['mapClass']

ProductID
1      noClass
2      noClass
3      noClass
4      noClass
316    noClass
        ...   
995     Medium
996       High
997        Low
998        Low
999        Low
Name: mapClass, Length: 504, dtype: object

In [57]:
prod = prod.replace({'nClass': class_dict})
prod['nClass']

ProductID
1      noClass
2      noClass
3      noClass
4      noClass
316    noClass
        ...   
995     Medium
996       High
997        Low
998        Low
999        Low
Name: nClass, Length: 504, dtype: object

In [58]:
prod['nClass'] = prod['nClass'].apply(product_class)
prod['nClass'].unique()

array(['N'], dtype=object)

**Exercise** What is the average price for Women's bikes?

In [59]:
prod[prod['nStyle']== 'W']['ListPrice'].mean() # gives out na, what is happening?

nan

In [60]:
#answer:
prod[prod['nStyle'] == 'W'] # let's just check if our filtering is working
prod['nStyle'].unique() # let's see the unique values
prod[prod['nStyle'] == 'W']
prod['nStyle'] = prod['nStyle'].str.strip()
prod['nStyle'].unique()
prod[prod['nStyle']== 'W']['ListPrice'].mean() # now it works


np.float64(726.779285714286)

**Exercise:** Now, get the mean price for Road Bikes.


In [61]:
prod['ProductLine'].unique()
prod['ProductLine'] = prod['ProductLine'].str.strip()
prod[prod['ProductLine']== 'R']['ListPrice'].mean()

np.float64(965.3488000000002)

In [62]:
prod['nProductLine'].str.strip()

ProductID
1      NoLine
2      NoLine
3      NoLine
4      NoLine
316    NoLine
        ...  
995    NoLine
996    NoLine
997      Road
998      Road
999      Road
Name: nProductLine, Length: 504, dtype: object

---
# Section 7: Detecting and Removing Duplicates
---

Duplicates can creep in from data joins, repeated entries, or import errors.

```python
df.duplicated()           # boolean Series, True for duplicates
df.duplicated().sum()     # how many?
df.drop_duplicates()      # remove duplicates (keeps first occurrence)
df.drop_duplicates(subset=['email'])    # only check certain columns
df.drop_duplicates(keep='last')          # keep the last occurrence instead
```

In [63]:
import pandas as pd

df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Alice', 'Carol', 'Bob'],
    'email': ['alice@x.com', 'bob@x.com', 'ALICE@x.com', 'carol@x.com', 'bob@x.com']
})

print('Duplicates:', df.duplicated().sum())

# After lowercasing email
df['email'] = df['email'].str.lower()
print('After lowercasing -- duplicates:', df.duplicated().sum())

df_clean = df.drop_duplicates()
print(df_clean)

Duplicates: 1
After lowercasing -- duplicates: 2
    name        email
0  Alice  alice@x.com
1    Bob    bob@x.com
3  Carol  carol@x.com


---
# Section 8: Exercises
---

## Exercise 1 -- Diagnose Missing Data

For the dataset below:
1. Print the count of missing values per column
2. Print the percentage of missing values per column
3. Drop columns with more than 50% missing
4. Fill remaining numerical missing with the column mean
5. Fill remaining categorical missing with the mode

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':   ['Alice', 'Bob', None, 'David', 'Eve'],
    'age':    [30, np.nan, 25, np.nan, 40],
    'salary': [50000, 60000, 45000, np.nan, 75000],
    'phone':  [None, None, None, '123', None],
    'city':   ['London', 'Leeds', 'London', None, 'Bristol']
})
df

# Your solution:


<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# 1
print(df.isnull().sum())

# 2
print((df.isnull().sum() / len(df) * 100).round(1))

# 3
threshold = 0.5
df = df.loc[:, df.isnull().mean() < threshold]
print('After dropping high-NaN cols:', df.columns.tolist())

# 4
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].mean())

# 5
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print(df)
```
</details>

## Exercise 2 -- Clean Customer Names

Given the messy customer names below:
1. Strip whitespace
2. Convert to title case
3. Remove duplicates after cleaning
4. Find any names with non-alphabetic characters

In [ ]:
import pandas as pd

names = pd.Series([
    '  alice smith  ',
    'BOB JONES',
    'Alice Smith',
    'Carol O\'Brien',
    '  carol o\'brien ',
    'David-Brown',
    'Eve123'
])

# Your solution:


<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# 1 + 2
cleaned = names.str.strip().str.title()
print('Cleaned:'); print(cleaned)

# 3
deduped = cleaned.drop_duplicates()
print('Unique:'); print(deduped)

# 4 -- non-alphabetic (allow spaces, hyphens, apostrophes)
import re
suspicious = deduped[~deduped.str.match(r'^[A-Za-z\' \-]+$')]
print('Suspicious:'); print(suspicious)
```
</details>

---
## Further Reading

- [Pandas user guide -- working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Pandas string methods](https://pandas.pydata.org/docs/user_guide/text.html)
- [Detecting outliers](https://towardsdatascience.com/detecting-and-handling-outliers)